In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/q21_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

# Pre-filter lineitems by orders with status 'F'
li = (lineitem[['L_ORDERKEY', 'L_SUPPKEY', 'L_RECEIPTDATE', 'L_COMMITDATE']]
      .merge(
          orders[orders.O_ORDERSTATUS == 'F'][['O_ORDERKEY']],
          left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner'
      )
      .drop('O_ORDERKEY', axis=1)
)

# Apply date filter
date_li = li[li.L_RECEIPTDATE > li.L_COMMITDATE][['L_ORDERKEY', 'L_SUPPKEY']]

# Compute supplier counts per order (original vs. after date filter)
orig_counts = li.groupby('L_ORDERKEY').L_SUPPKEY.nunique().reset_index(name='orig_count')
new_counts  = date_li.groupby('L_ORDERKEY').L_SUPPKEY.nunique().reset_index(name='new_count')

# Select orders with >1 original supplier and exactly one after filter
valid_orders = (
    orig_counts.merge(new_counts, on='L_ORDERKEY')
               .query('orig_count > 1 and new_count == 1')
               [['L_ORDERKEY']]
)

# Keep only qualifying lineitems
filtered_li = date_li.merge(valid_orders, on='L_ORDERKEY', how='inner')

# Build supplier → Saudi Arabia lookup
supp_nat = (
    supplier[['S_SUPPKEY', 'S_NATIONKEY', 'S_NAME']]
    .merge(
        nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']],
        left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner'
    )
    [['S_SUPPKEY', 'S_NAME']]
)

# Count waiting lineitems per supplier and join names
total = (
    filtered_li
    .groupby('L_SUPPKEY')
    .size()
    .reset_index(name='NUMWAIT')
    .merge(supp_nat, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    [['S_NAME', 'NUMWAIT']]
    .sort_values(['NUMWAIT', 'S_NAME'], ascending=[False, True])
)